# 01 — Análise Exploratória

Grupo 1 — Breno Luna e Paula Carlesso

Retoma a EDA da Aula 08 com foco no que o modelo vai usar. Checklist:

- [ ] Volume, período e integridade das seis tabelas
- [ ] Grau de desbalanceamento do rótulo (contestação)
- [ ] Valor: distribuição geral e por classe
- [ ] Hora do dia, canal e categoria de lojista
- [ ] Padrão de teste de cartão


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ARQUIVO_DATASET, COLUNA_ALVO, FIGURES_DIR

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

## 1. Carregar as tabelas brutas

In [ ]:
from src.etl.extract import carregar_tabelas

dfs = carregar_tabelas()
{nome: df.shape for nome, df in dfs.items()}

## 2. Quarentena — o que seria descartado, e por quê

In [ ]:
from src.etl.transform import separar_quarentena

validas, rejeitadas = separar_quarentena(dfs["transacoes"])
rejeitadas["motivo_rejeicao"].value_counts()

## 3. O rótulo

In [ ]:
from src.etl.build_dataset import adicionar_rotulo

base = adicionar_rotulo(validas, dfs["contestacoes"])
print(base[COLUNA_ALVO].value_counts())
print((base[COLUNA_ALVO].value_counts(normalize=True) * 100).round(3))

Se a proporção de fraude ficar na casa de 1%, acurácia está descartada
como métrica. O foco passa a ser recall, precision e AUC-PR.

## 4. Valor por classe

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(base.loc[base[COLUNA_ALVO] == 0, "valor"], bins=50, ax=eixos[0]).set_title("Legítimas")
sns.histplot(base.loc[base[COLUNA_ALVO] == 1, "valor"], bins=50, ax=eixos[1], color="crimson").set_title("Contestadas")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribuicao-valor-por-classe.png", dpi=150, bbox_inches="tight")

## 5. Taxa de fraude por hora do dia

In [ ]:
base["hora"] = pd.to_datetime(base["data_hora"]).dt.hour
taxa = base.groupby("hora")[COLUNA_ALVO].mean() * 100
ax = taxa.plot(kind="bar", figsize=(10, 4), title="Taxa de fraude (%) por hora")
plt.savefig(FIGURES_DIR / "taxa-fraude-por-hora.png", dpi=150, bbox_inches="tight")

## 6. Canal e categoria

In [ ]:
from src.etl.build_dataset import unificar

unificada = unificar(dfs, validas)
unificada = adicionar_rotulo(unificada, dfs["contestacoes"])

print(unificada.groupby("canal")[COLUNA_ALVO].agg(["mean", "count"]))
unificada.groupby("nome_categoria")[COLUNA_ALVO].mean().sort_values(ascending=False).head(10)

## 7. Conclusões

_Registrar aqui o que orienta as features e a modelagem._